# The three Jev questions

Jev does not write a reply. This notebook sends one shopper message and three questions in a single call.

- **Noul** — does this need someone now?
- **Choice** — which team should take it?
- **Score** — how bad is it?

The message is Maya Chen's duplicate charge, ticket `T-104`, from the sample shop.


In [3]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


Jev model: jev-latest
Jev key set: True
OpenAI key set: True


## Urgent, team, and severity

One state, three questions. The keys (`urgent`, `team`, `severity`) are for our code. Jev only sees `instructions`.


In [4]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    state = ticket("T-104")["body"]
    print(state)
    questions = {
        "urgent": Noul(instructions="Does this message need attention right now?"),
        "team": Choice(
            instructions="Which team should handle this message?",
            criteria={
                "billing": "Charges, refunds, invoices, or duplicate payments",
                "shipping": "Where an order is, or a late delivery",
                "account": "Login, password, or profile access",
            },
        ),
        "severity": Score(
            instructions="How severe is the problem for the shopper?",
            criteria=["Cosmetic", "Annoying but they can wait", "Blocking, they want it fixed now"],
        ),
    }
    response = ask(state, questions)
    show(response)
    print("team:", response.choices["team"].choice)
    print("urgent probability:", round(response.nouls["urgent"].noul, 2))
    print("severity:", round(response.scores["severity"].score, 2))


I was charged twice for order A-104. Both charges are $49. I want the extra charge refunded today.


TypeSafeNotFoundError: POST https://api.typesafe.ai/v1/v1/systemone/v1/systemone: 404 Not Found (request_id=req_01a0cb42863773cf871a7b611a28ea77)

**What you should see.** Billing should win, urgency should be high, and severity should sit toward the blocking end of the scale. Exact decimals move a little between runs.
